In [0]:
from pyspark.sql.functions import *

In [0]:
SOURCE_TABLE = "workspace.silver.historical_hourly"

TARGET_TABLE = "workspace.gold.daily_weather_summary"

In [0]:
# Read Silver

silver_df = spark.table(SOURCE_TABLE)

In [0]:
# Daily Metric

daily_summary_df = (
    silver_df
    .groupBy(
        "city_name",
        "year",
        "month",
        "day"
    )
    .agg(
        avg("temperature").alias("avg_temperature"),
        max("temperature").alias("max_temperature"),
        min("temperature").alias("min_temperature"),

        avg("humidity").alias("avg_humidity"),

        avg("wind_speed").alias("avg_wind_speed"),
        max("wind_speed").alias("max_wind_speed"),

        sum("precipitation_lwe").alias("total_precipitation"),

        avg("visibility").alias("avg_visibility"),

        avg("index_uv").alias("avg_uv_index"),

        avg("cloud_cover_total").alias("avg_cloud_cover"),

        count("*").alias("hourly_observations")
    )
)

In [0]:
# Clean

daily_summary_df = (
    daily_summary_df

    .withColumn(
        "avg_temperature",
        round(col("avg_temperature"),2)
    )

    .withColumn(
        "avg_humidity",
        round(col("avg_humidity"),2)
    )

    .withColumn(
        "avg_wind_speed",
        round(col("avg_wind_speed"),2)
    )

    .withColumn(
        "avg_visibility",
        round(col("avg_visibility"),2)
    )

    .withColumn(
        "avg_uv_index",
        round(col("avg_uv_index"),2)
    )

    .withColumn(
        "avg_cloud_cover",
        round(col("avg_cloud_cover"),2)
    )

    .withColumn(
        "total_precipitation",
        round(col("total_precipitation"),2)
    )
)

In [0]:
daily_summary_df = (
    daily_summary_df
    .withColumn(
        "date",
        to_date(
            concat_ws(
                "-",
                col("year"),
                col("month"),
                col("day")
            )
        )
    )
)

In [0]:
display(daily_summary_df)

daily_summary_df.printSchema()

print(f"Rows : {daily_summary_df.count()}")

In [0]:
(
    daily_summary_dfworkspace.gold.city_weather_summary.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET_TABLE)
)

In [0]:
print(silver_df.columns)